Rules

Only exclude the edge if all three are true:
1. Same subject entity
2. Same object entity
3. Predicate is a paraphrase of an existing predicate
(embedding similarity ≥ 0.9)
AND
You have already recorded the surface form elsewhere

else accept the edge.

| Case                                | Action                   |
| ----------------------------------- | ------------------------ |
| Same entities, exact same predicate | Exclude (true duplicate) |
| Same entities, paraphrase predicate | Accept + normalize       |
| Same entities, broader predicate    | Accept                   |
| Same entities, different relation   | Accept                   |
| Same entities, conflicting relation | Accept + flag            |


My current entity resolution pipeline

A. Blocking (Generate Candidates)
- FAISS based blocking

B. Entity Resolution (Check similarity)
- String similarity
- Embedding similarity 
- Neighborhood similarity

C. Merging (Merge similar entities)
- Merge entities threshold is met
 

Improvements (in priority order)
1. Predicate canonicalization
This will:

- Improve neighbor overlap
- Reduce graph fragmentation
- Improve query quality

2. Confidence-weighted merging

Instead of deciding to MERGE or NO MERGE

Use a confidence weighted merging e.g(MERGE (0.92 confidence))
This unlocks:
- Auditing
- Rollbacks
- Batch refinement

In [23]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, util

from generate_graph import get_propositions, generateEdges, createGraph, get_propositions_nosplit
from refine_graph_v4 import find_merge_candidates
from query_graph import QueryGraph
from tqdm import tqdm
from fuzzywuzzy import fuzz
from knowledge_graph_maker import Edge, Node

import pandas as pd
tqdm.pandas()

In [24]:
# --- Initialize embedding model ---
# model = SentenceTransformer("all-MiniLM-L6-v2")

model = SentenceTransformer("all-mpnet-base-v2")

# model = SentenceTransformer("E5-Base")


In [31]:
from knowledge_graph_maker import Edge, Node

list_of_edges2 = [Edge(node_1=Node(label='Person', name='John Doe', properties={'occupation': 'software engineer', 'rank': 'employee', 'birth_place': ''}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'John Doe works at Microsoft Corporation as a software engineer.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=0),
 Edge(node_1=Node(label='Person', name='J. Doe', properties={'occupation': 'employee'}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'J. Doe is an employee of Microsoft.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=1),
 
 Edge(node_1=Node(label='Person', name='Alice Smith', properties={'occupation': 'Employee', 'rank': 'N/A', 'birth_place': 'N/A'}), node_2=Node(label='Organization', name='Google LLC', properties={}), relationship='employee of', metadata={'summary': 'Alice Smith is employed by Google LLC.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=2),
 
 Edge(node_1=Node(label='Person', name='A. Smith', properties={'occupation': 'Employee'}), node_2=Node(label='Organization', name='Google', properties={}), relationship='works at', metadata={'summary': 'A. Smith works at Google.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=3),
 
 Edge(node_1=Node(label='Organization', name='Amazon', properties={}), node_2=Node(label='Object', name='Alexa', properties={}), relationship='develops', metadata={'summary': 'Amazon develops the voice assistant Alexa.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=4),
 
 Edge(node_1=Node(label='Organization', name='Amazon Inc.', properties={}), node_2=Node(label='Object', name='Alexa voice assistant', properties={}), relationship='produces', metadata={'summary': 'Amazon Inc. produces the Alexa voice assistant.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=5),
 
 Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': 'Pretoria, South Africa'}), node_2=Node(label='Organization', name='SpaceX', properties={'type': 'Aerospace manufacturer', 'founded': '2002'}), relationship='leads', metadata={'summary': 'Elon Musk leads SpaceX.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=6),
 
 Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': '', 'rank': ''}), node_2=Node(label='Organization', name='Space Exploration Technologies', properties={}), relationship='CEO of', metadata={'summary': 'Elon Musk is the CEO of Space Exploration Technologies.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=7),
 
 Edge(node_1=Node(label='Organization', name='Apple', properties={}), node_2=Node(label='Person', name='Steve Jobs', properties={'occupation': 'Entrepreneur', 'rank': 'Founder'}), relationship='founded by', metadata={'summary': 'Apple was founded by Steve Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=8),
 
 Edge(node_1=Node(label='Organization', name='Apple Inc.', properties={}), node_2=Node(label='Person', name='S. Jobs', properties={}), relationship='co-founded by', metadata={'summary': 'Apple Inc. was co-founded by S. Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=9)]


toy_data = []
for edge in list_of_edges2:
# unique_entities.add(edge.node_1.name)
# unique_entities.add(edge.node_2.name)

    toy_data.append({
        "subject": edge.node_1.name,
        "predicate": edge.relationship,
        "object": edge.node_2.name     
        # "context": edge.metadata.get("summary"),
    })

## V1

In [32]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "J. Doe", "predicate": "employee of", "object": "Microsoft"},
    {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# 1: Helper functions
# ----------------------------

def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# 2: Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# 3: Semantic blocking (toy: all existing entities)
# ----------------------------
def semantic_blocking(entity_text, entity_store, threshold=0.7):
    """Return candidate entity IDs if similarity > threshold"""
    candidates = []
    for eid, info in entity_store.items():
        score = string_sim(entity_text, info["name"])
        if score >= threshold:
            print("candidate found:", eid, info["name"], "score:", score)
            candidates.append({"entity_id": eid, "score": score})
    return candidates

# ----------------------------
# 4: Pre-insertion filtering
# ----------------------------
def pre_insertion_filtering(mention_text, candidates, threshold=0.75):
    """Return the best candidate if score >= threshold, else None"""
    best_score = 0
    best_entity_id = None
    for c in candidates:
        if c["score"] > best_score:
            best_score = c["score"]
            best_entity_id = c["entity_id"]
    if best_score >= threshold:
        return best_entity_id
    return None

def neighbor_overlap(new_neighbors, existing_neighbors):
    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)
    return len(intersection) / len(union)

# ----------------------------
# 5: Resolve entity (MERGE or INSERT)
# ----------------------------
def resolve_entity(entity_text):
    global entity_id_counter
    # blocking
    candidates = semantic_blocking(entity_text, entity_store)
    # filtering
    resolved_id = pre_insertion_filtering(entity_text, candidates)
    if resolved_id is not None:
        return resolved_id, "MERGE"
    # INSERT new entity
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {"name": entity_text, "neighbors": set()}
    entity_ids.append(new_id)
    return new_id, "INSERT"

# ----------------------------
# 6: Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")

# ----------------------------
# 7: Run the pipeline
# ----------------------------
for triple in toy_dataset:
    mentions = extract_entities(triple)
    resolved = {}
    actions = {}

    print("========================")
    # resolve each entity
    for ent in mentions:
        print("Resolving entity:", ent["text"])
        eid, action = resolve_entity(ent["text"])
        resolved[ent["role"]] = eid
        actions[ent["role"]] = action

    # create edge key to check duplicates
    edge_key = (resolved["subject"], triple["predicate"], resolved["object"])
    print("existing_edge_keys:", existing_edge_keys)
    print("edge_key:", edge_key)
    if edge_key in existing_edge_keys:
        print("dup")
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": resolved["subject"],
            "object_id": resolved["object"]
        })
    else:
        accepted_edges.append({
            **triple,
            "subject_id": resolved["subject"],
            "object_id": resolved["object"],
            "subject_action": actions["subject"],
            "object_action": actions["object"]
        })
        existing_edge_keys.add(edge_key)
        # update neighbors
        insert_edge(resolved["subject"], triple["predicate"], resolved["object"])


# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info)


Resolving entity: John Doe
Resolving entity: Microsoft
existing_edge_keys: set()
edge_key: ('E1', 'works at', 'E2')
Resolving entity: J. Doe
candidate found: E1 John Doe score: 0.77
Resolving entity: Microsoft
candidate found: E2 Microsoft score: 1.0
existing_edge_keys: {('E1', 'works at', 'E2')}
edge_key: ('E1', 'employee of', 'E2')
Resolving entity: Alice Smith
Resolving entity: Google
existing_edge_keys: {('E1', 'employee of', 'E2'), ('E1', 'works at', 'E2')}
edge_key: ('E3', 'works at', 'E4')
Resolving entity: A. Smith
candidate found: E3 Alice Smith score: 0.78
Resolving entity: Google
candidate found: E4 Google score: 1.0
existing_edge_keys: {('E3', 'works at', 'E4'), ('E1', 'employee of', 'E2'), ('E1', 'works at', 'E2')}
edge_key: ('E3', 'works at', 'E4')
dup
Resolving entity: Amazon
Resolving entity: Alexa
existing_edge_keys: {('E3', 'works at', 'E4'), ('E1', 'employee of', 'E2'), ('E1', 'works at', 'E2')}
edge_key: ('E5', 'develops', 'E6')
Resolving entity: Amazon Inc.
candida

## V2

In [33]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Helper functions
# ----------------------------

def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# Semantic blocking (toy: all existing entities)
# ----------------------------
def semantic_blocking(entity_text, entity_store, threshold=0.7):
    """Return candidate entity IDs if similarity > threshold"""
    candidates = []
    for eid, info in entity_store.items():
        score = string_sim(entity_text, info["name"])
        if score >= threshold:
            print("semantic_blocking")
            print("candidate found:", eid, info["name"], "score:", score)
            candidates.append({"entity_id": eid, "score": score})
    return candidates


def neighbor_overlap(new_neighbors, existing_neighbors):

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

# ----------------------------
# Resolve entity (MERGE or INSERT)
# ----------------------------

def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    """
    role: 'subject' or 'object'
    predicate: predicate of the current triple
    other_entity_id: resolved ID of the other entity in the triple (if exists)
    """
    global entity_id_counter

    # Blocking
    candidates = semantic_blocking(entity_text, entity_store)

    best_candidate = None
    best_score = 0

    for c in candidates:
        existing = entity_store[c["entity_id"]]

        # String similarity
        s_sim = string_sim(entity_text, existing["name"])

        # Neighbor similarity (if possible)
        new_neighbors = set()
        if other_entity_id is not None:
            new_neighbors.add(f"{predicate}:{other_entity_id}")

        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        print(f"Evaluating candidate {c['entity_id']}: s_sim={s_sim}, n_sim={n_sim}")
        # Merge decision (gated)
        if (s_sim >= 0.9 or (s_sim >= 0.75 and n_sim >= 0.5)):
            if s_sim > best_score:
                best_candidate = c["entity_id"]
                best_score = s_sim

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {
        "name": entity_text,
        "neighbors": set()
    }
    return new_id, "INSERT"

# ----------------------------
# Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info)


Resolving triple: {'subject': 'John Doe', 'predicate': 'works at', 'object': 'Microsoft'}
Resolving triple: {'subject': 'J. Doe', 'predicate': 'works at', 'object': 'Microsoft'}
semantic_blocking
candidate found: E1 John Doe score: 0.77
new_neighbors: set()
existing_neighbors: {'works at:E2'}
Evaluating candidate E1: s_sim=0.77, n_sim=0.0
semantic_blocking
candidate found: E2 Microsoft score: 1.0
new_neighbors: {'works at:E3'}
existing_neighbors: {'works at:E1'}
Evaluating candidate E2: s_sim=1.0, n_sim=0.0
Resolving triple: {'subject': 'Alice Smith', 'predicate': 'works at', 'object': 'Google'}
Resolving triple: {'subject': 'A. Smith', 'predicate': 'works at', 'object': 'Google'}
semantic_blocking
candidate found: E4 Alice Smith score: 0.78
new_neighbors: set()
existing_neighbors: {'works at:E5'}
Evaluating candidate E4: s_sim=0.78, n_sim=0.0
semantic_blocking
candidate found: E5 Google score: 1.0
new_neighbors: {'works at:E6'}
existing_neighbors: {'works at:E4'}
Evaluating candidate 

## V3

In [34]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Helper functions
# ----------------------------
def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def embedding_sim(a, b):
    """Cosine similarity between two strings"""
    emb_a = model.encode(a, convert_to_tensor=True)
    emb_b = model.encode(b, convert_to_tensor=True)
    return util.cos_sim(emb_a, emb_b).item()

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# Semantic blocking (toy: all existing entities)
# ----------------------------

def semantic_blocking(entity_text, entity_store, threshold=0.6):
    """
    Hybrid blocking using string + embedding similarity
    """
    candidates = []
    query_emb = model.encode(entity_text, convert_to_tensor=True)

    for eid, info in entity_store.items():
        s_sim = string_sim(entity_text, info["name"])
        e_sim = util.cos_sim(query_emb, info["embedding"]).item()

        # loose blocking, strict merge later
        if max(s_sim, e_sim) >= threshold:
            print("semantic_blocking")
            print(
                f"candidate: {eid}, name={info['name']}, "
                f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}"
            )
            candidates.append({
                "entity_id": eid,
                "s_sim": s_sim,
                "e_sim": e_sim
            })
    return candidates



def neighbor_overlap(new_neighbors, existing_neighbors):

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

# ----------------------------
# Resolve entity (MERGE or INSERT)
# ----------------------------

def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    global entity_id_counter

    candidates = semantic_blocking(entity_text, entity_store)

    best_candidate = None
    best_score = 0

    for c in candidates:
        existing = entity_store[c["entity_id"]]

        s_sim = c["s_sim"]
        e_sim = c["e_sim"]

        # Neighbor similarity
        new_neighbors = set()
        if other_entity_id is not None:
            new_neighbors.add(f"{predicate}:{other_entity_id}")

        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        # Weighted fusion score
        fused_score = 0.4 * s_sim + 0.4 * e_sim + 0.2 * n_sim

        print(
            f"Evaluating {c['entity_id']}: "
            f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}, "
            f"n_sim={n_sim:.3f}, fused={fused_score:.3f}"
        )

        # Merge rules (interpretable + safe)
        if (e_sim >= 0.85 or (s_sim >= 0.8 and e_sim >= 0.7) or (fused_score >= 0.75)
        ):
            if fused_score > best_score:
                best_candidate = c["entity_id"]
                best_score = fused_score

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1

    entity_store[new_id] = {
        "name": entity_text,
        "embedding": model.encode(entity_text, convert_to_tensor=True),
        "neighbors": set()
    }

    return new_id, "INSERT"

# ----------------------------
# Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

# print("\n=== Entity Store ===")
# for eid, info in entity_store.items():
#     print(eid, info)


Resolving triple: {'subject': 'John Doe', 'predicate': 'works at', 'object': 'Microsoft'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

semantic_blocking
candidate: E15, name=Steve Jobs, s_sim=0.750, e_sim=0.408
new_neighbors: {'co-founded by:E16'}
existing_neighbors: {'founded by:E14'}
Evaluating E15: s_sim=0.750, e_sim=0.408, n_sim=0.000, fused=0.463


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Accepted Edges ===
{'subject': 'John Doe', 'predicate': 'works at', 'object': 'Microsoft', 'subject_id': 'E1', 'object_id': 'E2', 'subject_action': 'INSERT', 'object_action': 'INSERT'}
{'subject': 'J. Doe', 'predicate': 'works at', 'object': 'Microsoft', 'subject_id': 'E3', 'object_id': 'E2', 'subject_action': 'INSERT', 'object_action': 'MERGE'}
{'subject': 'Alice Smith', 'predicate': 'works at', 'object': 'Google', 'subject_id': 'E4', 'object_id': 'E5', 'subject_action': 'INSERT', 'object_action': 'INSERT'}
{'subject': 'A. Smith', 'predicate': 'works at', 'object': 'Google', 'subject_id': 'E6', 'object_id': 'E5', 'subject_action': 'INSERT', 'object_action': 'MERGE'}
{'subject': 'Amazon', 'predicate': 'develops', 'object': 'Alexa', 'subject_id': 'E7', 'object_id': 'E8', 'subject_action': 'INSERT', 'object_action': 'INSERT'}
{'subject': 'Amazon Inc.', 'predicate': 'produces', 'object': 'Alexa voice assistant', 'subject_id': 'E9', 'object_id': 'E10', 'subject_action': 'INSERT', 'obje

## V4

In [26]:
EMBED_DIM = 768  # MiniLM; use 768 for mpnet

faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_id_map = []  # maps FAISS row → entity_id

# ----------------------------
# Helper functions
# ----------------------------

def normalize(v):
    return v / np.linalg.norm(v)


def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def embedding_sim(a, b):
    """Cosine similarity between two strings"""
    emb_a = model.encode(a, convert_to_tensor=True)
    emb_b = model.encode(b, convert_to_tensor=True)
    return util.cos_sim(emb_a, emb_b).item()

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# Semantic blocking (toy: all existing entities)
# ----------------------------

# def semantic_blocking(entity_text, entity_store, threshold=0.6):
#     """
#     Hybrid blocking using string + embedding similarity
#     """
#     candidates = []
#     query_emb = model.encode(entity_text, convert_to_tensor=True)

#     for eid, info in entity_store.items():
#         s_sim = string_sim(entity_text, info["name"])
#         e_sim = util.cos_sim(query_emb, info["embedding"]).item()

#         # loose blocking, strict merge later
#         if max(s_sim, e_sim) >= threshold:
#             print("semantic_blocking")
#             print(
#                 f"candidate: {eid}, name={info['name']}, "
#                 f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}"
#             )
#             candidates.append({
#                 "entity_id": eid,
#                 "s_sim": s_sim,
#                 "e_sim": e_sim
#             })
#     return candidates

def semantic_blocking_faiss(entity_text, top_k=10, sim_threshold=0.7):
    if faiss_index.ntotal == 0:
        return []

    query_emb = model.encode(entity_text)
    query_emb = normalize(query_emb).reshape(1, -1)

    scores, indices = faiss_index.search(query_emb, top_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        if score >= sim_threshold:
            eid = faiss_id_map[idx]
            candidates.append({
                "entity_id": eid,
                "e_sim": float(score)
            })

    return candidates


def neighbor_overlap(new_neighbors, existing_neighbors):

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

# ----------------------------
# Resolve entity (MERGE or INSERT)
# ----------------------------

def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    global entity_id_counter

    candidates = semantic_blocking_faiss(entity_text)

    best_candidate = None
    best_score = 0

    for c in candidates:
        existing = entity_store[c["entity_id"]]

        s_sim = string_sim(entity_text, existing["name"])
        e_sim = c["e_sim"]

        new_neighbors = set()
        if other_entity_id:
            new_neighbors.add(f"{predicate}:{other_entity_id}")

        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        fused_score = 0.4 * s_sim + 0.4 * e_sim + 0.2 * n_sim

        print(
            f"Evaluating {c['entity_id']}: "
            f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}, "
            f"n_sim={n_sim:.3f}, fused={fused_score:.3f}"
        )

        # Merge rules (interpretable + safe)
        if (e_sim >= 0.85 or (s_sim >= 0.8 and e_sim >= 0.7) or (fused_score >= 0.75)
        ):
            if fused_score > best_score:
                best_candidate = c["entity_id"]
                best_score = fused_score

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1

    embedding = model.encode(entity_text)
    embedding = normalize(embedding)

    faiss_index.add(embedding.reshape(1, -1))
    faiss_id_map.append(new_id)

    entity_store[new_id] = {
        "name": entity_text,
        "embedding": embedding,
        "neighbors": set()
    }

    return new_id, "INSERT"

# ----------------------------
# Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")


### Toy dataset

In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    # {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    # {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    # {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    # {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    # {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    # {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "E. Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "E. Musk", "predicate": "founder of", "object": "SpaceX"},
    # {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    # {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info['name'], info['neighbors'])

### Stress test dataset

In [30]:
stress_test_dataset_with_labels = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    {"subject": "Apple", "predicate": "released", "object": "iPhone 15", "gold_cluster": "APPLE_TECH"},
    {"subject": "Apple", "predicate": "harvested in", "object": "September", "gold_cluster": "APPLE_FRUIT"},
    {"subject": "Apple", "predicate": "is a type of", "object": "Fruit", "gold_cluster": "APPLE_FRUIT"},

    # =========================
    # 2. Abbreviation Ambiguity
    # =========================
    {"subject": "US", "predicate": "invaded", "object": "Iraq", "gold_cluster": "UNITED_STATES"},
    {"subject": "US", "predicate": "won", "object": "US Open", "gold_cluster": "US_TENNIS_PLAYER"},
    {"subject": "United States", "predicate": "has capital", "object": "Washington DC", "gold_cluster": "UNITED_STATES"},

    # =========================
    # 3. Semantic Embedding Trap
    # =========================
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa", "gold_cluster": "AMAZON_COMPANY"},
    {"subject": "Amazon", "predicate": "headquartered in", "object": "Seattle", "gold_cluster": "AMAZON_COMPANY"},
    {"subject": "Amazon River", "predicate": "flows through", "object": "Brazil", "gold_cluster": "AMAZON_RIVER"},

    # =========================
    # 4. Role vs Entity Confusion
    # =========================
    {"subject": "President", "predicate": "leads", "object": "United States", "gold_cluster": "US_PRESIDENT_ROLE"},
    {"subject": "Joe Biden", "predicate": "is", "object": "President", "gold_cluster": "JOE_BIDEN"},
    {"subject": "Joe Biden", "predicate": "born in", "object": "Scranton", "gold_cluster": "JOE_BIDEN"},

    # =========================
    # 5. Predicate Paraphrase
    # =========================
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX", "gold_cluster": "ELON_MUSK"},
    {"subject": "Elon Musk", "predicate": "is CEO of", "object": "SpaceX", "gold_cluster": "ELON_MUSK"},
    {"subject": "Elon Musk", "predicate": "runs", "object": "SpaceX", "gold_cluster": "ELON_MUSK"},

    # =========================
    # 6. Predicate Polarity Conflict
    # =========================
    {"subject": "Company X", "predicate": "acquired", "object": "Company Y", "gold_cluster": "COMPANY_X"},
    {"subject": "Company Y", "predicate": "acquired", "object": "Company X", "gold_cluster": "COMPANY_Y"},

    # =========================
    # 7. Structural / Neighborhood Deception
    # =========================
    {"subject": "Paris", "predicate": "located in", "object": "France", "gold_cluster": "PARIS_FRANCE"},
    {"subject": "Paris", "predicate": "has river", "object": "Seine", "gold_cluster": "PARIS_FRANCE"},
    {"subject": "Paris", "predicate": "located in", "object": "Texas", "gold_cluster": "PARIS_TEXAS"},
    {"subject": "Paris", "predicate": "has population", "object": "25000", "gold_cluster": "PARIS_TEXAS"},

    # =========================
    # 8. Copycat Organization
    # =========================
    {"subject": "OpenAI", "predicate": "develops", "object": "ChatGPT", "gold_cluster": "OPENAI"},
    {"subject": "OpenAI Research", "predicate": "develops", "object": "ChatGPT", "gold_cluster": "OPENAI_RESEARCH"},
    {"subject": "OpenAI Research", "predicate": "published", "object": "Paper A", "gold_cluster": "OPENAI_RESEARCH"},
    {"subject": "OpenAI", "predicate": "published", "object": "Paper B", "gold_cluster": "OPENAI"},

    # =========================
    # 9. Temporal Drift
    # =========================
    {"subject": "Google", "predicate": "CEO", "object": "Larry Page", "gold_cluster": "GOOGLE"},
    {"subject": "Google", "predicate": "CEO", "object": "Sundar Pichai", "gold_cluster": "GOOGLE"},

    # =========================
    # 10. Compound Adversarial Case
    # =========================
    {"subject": "Meta", "predicate": "formerly known as", "object": "Facebook", "gold_cluster": "META"},
    {"subject": "Facebook", "predicate": "owns", "object": "Instagram", "gold_cluster": "META"},
    {"subject": "Meta AI", "predicate": "develops", "object": "LLaMA", "gold_cluster": "META_AI"},
    {"subject": "Facebook AI Research", "predicate": "develops", "object": "LLaMA", "gold_cluster": "FAIR"},
]


stress_test_dataset = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    {"subject": "Apple", "predicate": "released", "object": "iPhone 15"},
    {"subject": "Apple", "predicate": "harvested in", "object": "September"},
    {"subject": "Apple", "predicate": "is a type of", "object": "Fruit"},

    # # =========================
    # # 2. Abbreviation Ambiguity
    # # =========================
    # {"subject": "US", "predicate": "invaded", "object": "Iraq"},
    # {"subject": "US", "predicate": "won", "object": "US Open"},
    # {"subject": "United States", "predicate": "has capital", "object": "Washington DC"},

    # # =========================
    # # 3. Semantic Embedding Trap
    # # =========================
    # {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    # {"subject": "Amazon", "predicate": "headquartered in", "object": "Seattle"},
    # {"subject": "Amazon River", "predicate": "flows through", "object": "Brazil"},

    # # =========================
    # # 4. Role vs Entity Confusion
    # # =========================
    # {"subject": "President", "predicate": "leads", "object": "United States"},
    # {"subject": "Joe Biden", "predicate": "is", "object": "President"},
    # {"subject": "Joe Biden", "predicate": "born in", "object": "Scranton"},

    # # =========================
    # # 5. Predicate Paraphrase
    # # =========================
    # {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    # {"subject": "Elon Musk", "predicate": "is CEO of", "object": "SpaceX"},
    # {"subject": "Elon Musk", "predicate": "runs", "object": "SpaceX"},

    # # =========================
    # # 6. Predicate Polarity Conflict
    # # =========================
    # {"subject": "Company X", "predicate": "acquired", "object": "Company Y"},
    # {"subject": "Company Y", "predicate": "acquired", "object": "Company X"},

    # # =========================
    # # 7. Structural / Neighborhood Deception
    # # =========================
    # {"subject": "Paris", "predicate": "located in", "object": "France"},
    # {"subject": "Paris", "predicate": "has river", "object": "Seine"},
    # {"subject": "Paris", "predicate": "located in", "object": "Texas"},
    # {"subject": "Paris", "predicate": "has population", "object": "25000"},

    # # =========================
    # # 8. Copycat Organization
    # # =========================
    # {"subject": "OpenAI", "predicate": "develops", "object": "ChatGPT"},
    # {"subject": "OpenAI Research", "predicate": "develops", "object": "ChatGPT"},
    # {"subject": "OpenAI Research", "predicate": "published", "object": "Paper A"},
    # {"subject": "OpenAI", "predicate": "published", "object": "Paper B"},

    # # =========================
    # # 9. Temporal Drift
    # # =========================
    # {"subject": "Google", "predicate": "CEO", "object": "Larry Page"},
    # {"subject": "Google", "predicate": "CEO", "object": "Sundar Pichai"},

    # # =========================
    # # 10. Compound Adversarial Case
    # # =========================
    # {"subject": "Meta", "predicate": "formerly known as", "object": "Facebook"},
    # {"subject": "Facebook", "predicate": "owns", "object": "Instagram"},
    # {"subject": "Meta AI", "predicate": "develops", "object": "LLaMA"},
    # {"subject": "Facebook AI Research", "predicate": "develops", "object": "LLaMA"},
]

for triple in stress_test_dataset_with_labels:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info['name'], info['neighbors'])


Resolving triple: {'subject': 'Apple', 'predicate': 'released', 'object': 'iPhone 15', 'gold_cluster': 'APPLE_TECH'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

new_neighbors: set()
existing_neighbors: {'is a type of:E4', 'released:E2', 'harvested in:E3'}
Evaluating E1: s_sim=1.000, e_sim=1.000, n_sim=0.000, fused=0.800


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

new_neighbors: {'released:E1'}
existing_neighbors: {'released:E1'}
Evaluating E2: s_sim=1.000, e_sim=1.000, n_sim=1.000, fused=1.000
Resolving triple: {'subject': 'Apple', 'predicate': 'harvested in', 'object': 'September', 'gold_cluster': 'APPLE_FRUIT'}


...

Resolving triple: {'subject': 'Facebook AI Research', 'predicate': 'develops', 'object': 'LLaMA', 'gold_cluster': 'FAIR'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

new_neighbors: {'develops:E38'}
existing_neighbors: {'develops:E36'}
Evaluating E37: s_sim=1.000, e_sim=1.000, n_sim=0.000, fused=0.800
=== Accepted Edges ===
{'subject': 'Apple', 'predicate': 'released', 'object': 'iPhone 15', 'subject_id': 'E1', 'object_id': 'E2', 'subject_action': 'INSERT', 'object_action': 'INSERT'}
{'subject': 'Apple', 'predicate': 'harvested in', 'object': 'September', 'subject_id': 'E1', 'object_id': 'E3', 'subject_action': 'MERGE', 'object_action': 'INSERT'}
{'subject': 'Apple', 'predicate': 'is a type of', 'object': 'Fruit', 'subject_id': 'E1', 'object_id': 'E4', 'subject_action': 'MERGE', 'object_action': 'INSERT'}
{'subject': 'US', 'predicate': 'invaded', 'object': 'Iraq', 'gold_cluster': 'UNITED_STATES', 'subject_id': 'E5', 'object_id': 'E6', 'subject_action': 'INSERT', 'object_action': 'INSERT'}
{'subject': 'US', 'predicate': 'won', 'object': 'US Open', 'gold_cluster': 'US_TENNIS_PLAYER', 'subject_id': 'E5', 'object_id': 'E7', 'subject_action': 'MERGE', 'o

Evaluation script: will try 

In [31]:
from itertools import combinations
from collections import defaultdict

# ----------------------------
# Helper: Build cluster mapping
# ----------------------------
def build_cluster_map(entity_store, toy_dataset):
    """
    Map each resolved entity ID to its gold cluster
    """
    gold_map = {}
    for triple in toy_dataset:
        # subject
        gold_map[triple["subject"]] = triple["gold_cluster"]
        # object
        gold_map[triple["object"]] = triple["gold_cluster"]
    return gold_map

# ----------------------------
# Helper: Generate all pairwise links for evaluation
# ----------------------------
def pairwise_links(items):
    """
    Returns all pairwise combinations (unordered) of items
    """
    return set(frozenset([a, b]) for a, b in combinations(items, 2))

# ----------------------------
# Evaluation function
# ----------------------------
def evaluate_er_pipeline(resolved_entities, toy_dataset):
    """
    resolved_entities: dict mapping entity_text -> resolved_entity_id
    toy_dataset: list of dicts with 'gold_cluster'
    """

    # Build gold clusters
    gold_clusters = defaultdict(set)
    for triple in toy_dataset:
        gold_clusters[triple["gold_cluster"]].add(triple["subject"])
        gold_clusters[triple["gold_cluster"]].add(triple["object"])

    # Build predicted clusters
    pred_clusters = defaultdict(set)
    for entity_text, eid in resolved_entities.items():
        pred_clusters[eid].add(entity_text)

    # Build pairwise links
    gold_links = set()
    for cluster_entities in gold_clusters.values():
        gold_links.update(pairwise_links(cluster_entities))

    pred_links = set()
    for cluster_entities in pred_clusters.values():
        pred_links.update(pairwise_links(cluster_entities))

    # Compute pairwise Precision / Recall / F1
    true_positive = len(gold_links & pred_links)
    false_positive = len(pred_links - gold_links)
    false_negative = len(gold_links - pred_links)

    precision = true_positive / (true_positive + false_positive + 1e-10)
    recall = true_positive / (true_positive + false_negative + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)

    print(f"Evaluation Results:")
    print(f"Pairwise Precision: {precision:.3f}")
    print(f"Pairwise Recall   : {recall:.3f}")
    print(f"Pairwise F1       : {f1:.3f}")
    print()
    
    # Optional: log problematic merges
    problem_merges = []
    for cluster in pred_clusters.values():
        # Check if merged entities come from multiple gold clusters
        golds = set(toy_dataset[i]["gold_cluster"] 
                    for i, t in enumerate(toy_dataset)
                    if t["subject"] in cluster or t["object"] in cluster)
        if len(golds) > 1:
            problem_merges.append({
                "predicted_cluster": cluster,
                "gold_clusters": golds
            })
    if problem_merges:
        print("Problematic merges detected:")
        for pm in problem_merges:
            print(pm)
    else:
        print("No problematic merges detected")


In [ ]:
## These are from the entity store after running the pipeline

## Example
# resolved_entities = {
#     "Apple": "E1",
#     "Apple Fruit": "E2",
#     "US": "E3",
#     "United States": "E3",
#     "Joe Biden": "E4",
#     "President": "E5",
# }

# Build resolved_entities mapping from your entity_store
resolved_entities = {}

for eid, info in entity_store.items():
    entity_name = info["name"]
    resolved_entities[entity_name] = eid

print(resolved_entities)


{'Apple': 'E1', 'iPhone 15': 'E2', 'September': 'E3', 'Fruit': 'E4', 'US': 'E5', 'Iraq': 'E6', 'US Open': 'E7', 'United States': 'E8', 'Washington DC': 'E9', 'Amazon': 'E10', 'Alexa': 'E11', 'Seattle': 'E12', 'Amazon River': 'E13', 'Brazil': 'E14', 'President': 'E15', 'Joe Biden': 'E16', 'Scranton': 'E17', 'Elon Musk': 'E18', 'SpaceX': 'E19', 'Company X': 'E20', 'Paris': 'E21', 'France': 'E22', 'Seine': 'E23', 'Texas': 'E24', '25000': 'E25', 'OpenAI': 'E26', 'ChatGPT': 'E27', 'OpenAI Research': 'E28', 'Paper A': 'E29', 'Google': 'E30', 'Larry Page': 'E31', 'Sundar Pichai': 'E32', 'Meta': 'E33', 'Facebook': 'E34', 'Instagram': 'E35', 'Meta AI': 'E36', 'LLaMA': 'E37', 'Facebook AI Research': 'E38'}


In [33]:
## This is the main function call to evaluate
evaluate_er_pipeline(resolved_entities, stress_test_dataset_with_labels)

Evaluation Results:
Pairwise Precision: 0.000
Pairwise Recall   : 0.000
Pairwise F1       : 0.000

Problematic merges detected:
{'predicted_cluster': {'Apple'}, 'gold_clusters': {'APPLE_TECH', 'APPLE_FRUIT'}}
{'predicted_cluster': {'US'}, 'gold_clusters': {'UNITED_STATES', 'US_TENNIS_PLAYER'}}
{'predicted_cluster': {'United States'}, 'gold_clusters': {'UNITED_STATES', 'US_PRESIDENT_ROLE'}}
{'predicted_cluster': {'President'}, 'gold_clusters': {'JOE_BIDEN', 'US_PRESIDENT_ROLE'}}
{'predicted_cluster': {'Company X'}, 'gold_clusters': {'COMPANY_X', 'COMPANY_Y'}}
{'predicted_cluster': {'Paris'}, 'gold_clusters': {'PARIS_FRANCE', 'PARIS_TEXAS'}}
{'predicted_cluster': {'ChatGPT'}, 'gold_clusters': {'OPENAI_RESEARCH', 'OPENAI'}}
{'predicted_cluster': {'LLaMA'}, 'gold_clusters': {'META_AI', 'FAIR'}}


# To do

- Check why neighbor similarity is not working (its working)
- Convert this into a class-based pipeline
- Add FAISS-based blocking (done)
- Add entity embedding updates after merge